# Module 5.2: FFT Frequency-Domain Vibration & Bearing Kinematics
### 🇵🇱 Analiza Drgań w Dziedzinie Częstotliwości (FFT) i Kinematyka Łożysk

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/m05_l02_fft_frequency_domain_vibration.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Decompose raw accelerometer signals with FFT, apply Hanning windowing to stop leakage, and isolate bearing outer race defect peaks.  
> 🇵🇱 **Streszczenie:** Rozłóż sygnały z akcelerometru za pomocą FFT, zastosuj okno Hanninga i zidentyfikuj piki uszkodzenia bieżni łożyska.

- **Estimated Time:** 25 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **Time Domain Blindness**: Microscopic bearing fatigue cracks generate high-frequency micro-shocks hidden under background unbalance and process noise.
2. **Hanning Windowing**: Abrupt sample boundaries cause artificial spectral leakage. Windowing smoothly tapers endpoints to zero, restoring sharp frequency resolution.
3. **Bearing Fault Kinematics**:
   - $1X$ Shaft Rotational Speed: Mass unbalance.
   - $2X$ Shaft Speed: Shaft angular misalignment.
   - $	ext{BPFO}$ (Outer Race): $f_{BPFO} = rac{N_{balls}}{2} f_{shaft} \left(1 - rac{d}{D}\coslphaight)$.


In [ ]:
# 1. Synthesize 2000 Hz Accelerometer Signal with Bearing BPFO Fault
np.random.seed(42)
Fs = 2000.0  # Sampling rate: 2000 Hz
T = 1.0 / Fs
N = 2048
time_axis = np.linspace(0.0, N*T, N, endpoint=False)

f_shaft = 30.0            # 30 Hz shaft rotational speed (1800 RPM)
f_misalignment = 60.0     # 2X misalignment harmonic
f_bpfo = 3.58 * f_shaft   # 107.4 Hz outer race defect peak

# Synthetic multi-harmonic signal
sig_1x = 2.0 * np.sin(2.0 * np.pi * f_shaft * time_axis)
sig_2x = 0.8 * np.sin(2.0 * np.pi * f_misalignment * time_axis)
sig_bpfo = 1.6 * np.sin(2.0 * np.pi * f_bpfo * time_axis)
noise = 0.6 * np.random.randn(N)

vibration_accel = sig_1x + sig_2x + sig_bpfo + noise
print(f"✓ Vibration telemetry synthesized: {N} samples at {Fs} Hz.")


In [ ]:
# 2. Compute Fast Fourier Transform (FFT) with Hanning Window
window = np.hanning(N)
windowed_signal = vibration_accel * window

fft_vals = np.fft.rfft(windowed_signal)
fft_freqs = np.fft.rfftfreq(N, T)
fft_mag = (2.0 / N) * np.abs(fft_vals)

fig, axes = plt.subplots(2, 1, figsize=(14, 6))

axes[0].plot(time_axis[:250], vibration_accel[:250], color='#0284c7', linewidth=1.2)
axes[0].set_title("Time Domain: Accelerometer Raw Vibration (Defect Inaudible & Masked by Noise)", fontweight='bold')
axes[0].set_ylabel("Accel [m/s²]")
axes[0].grid(True, alpha=0.3)

axes[1].plot(fft_freqs, fft_mag, color='#10b981', linewidth=1.6, label='FFT Magnitude Spectrum')
axes[1].axvline(f_shaft, color='#f59e0b', linestyle='--', linewidth=1.8, label=f'1X Shaft Unbalance ({f_shaft:.1f} Hz)')
axes[1].axvline(f_misalignment, color='#6366f1', linestyle=':', linewidth=1.8, label=f'2X Misalignment ({f_misalignment:.1f} Hz)')
axes[1].axvline(f_bpfo, color='#ef4444', linestyle='-', linewidth=2.2, label=f'BPFO Outer Race Defect ({f_bpfo:.1f} Hz)')
axes[1].set_xlim(0, 250)
axes[1].set_title("Frequency Domain: FFT Exposes Incipient Bearing Defect Peak!", fontweight='bold')
axes[1].set_xlabel("Frequency [Hz]")
axes[1].set_ylabel("Vibration Amplitude [m/s²]")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Synthesize multi-harmonic vibration signal with bearing defect frequencies *(🇵🇱 Wygeneruj sygnał drgań z harmonicznymi i częstotliwością uszkodzenia łożyska)*
- [ ] **Task 2:** Apply Hanning window function to suppress spectral boundary leakage *(🇵🇱 Zastosuj funkcję okna Hanninga, aby stłumić przeciek widmowy na brzegach)*
- [ ] **Task 3:** Calculate Fast Fourier Transform (FFT) and detect Ball Pass Frequency Outer Race (BPFO) peak *(🇵🇱 Oblicz szybką transformatę Fouriera (FFT) i zlokalizuj pik częstotliwości BPFO)*


In [ ]:
# Exercise Challenge:
# Identify the peak frequency between 95 Hz and 120 Hz
bpfo_band = (fft_freqs >= 95.0) & (fft_freqs <= 120.0)
peak_detected_freq = fft_freqs[bpfo_band][np.argmax(fft_mag[bpfo_band])]
print(f"Detected Bearing Defect Frequency Peak: {peak_detected_freq:.2f} Hz (Theoretical: {f_bpfo:.2f} Hz)")
assert abs(peak_detected_freq - f_bpfo) < 1.5, "FFT defect peak detection out of tolerance!"
print("✓ FFT frequency domain bearing kinematics verified.")
